# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [6]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [5]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [7]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

In [2]:
# Some lists!

todos = []
completed = []

In [8]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [9]:
get_todo_report()

''

In [10]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [11]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [12]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [13]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [14]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [15]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [16]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [17]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [18]:
def loop(messages):
    done = False
    while not done:
        response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [19]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [20]:
todos, completed = [], []
loop(messages)

Todo #1: Estimate the distance between Boston and New York.
Todo #2: Calculate the position of the Boston train when the New York train starts.
Todo #3: Calculate the remaining distance between the two trains at 3:00 pm.
Todo #4: Calculate the relative speed of the two trains.
Todo #5: Calculate the time it takes for the trains to close the remaining distance.
Todo #6: Determine the time the trains meet.

The standard distance between Boston and New York City by rail is approximately 215 miles. I will use 215 miles for
this calculation.

Todo #1: Estimate the distance between Boston and New York.
Todo #2: Calculate the position of the Boston train when the New York train starts.
Todo #3: Calculate the remaining distance between the two trains at 3:00 pm.
Todo #4: Calculate the relative speed of the two trains.
Todo #5: Calculate the time it takes for the trains to close the remaining distance.
Todo #6: Determine the time the trains meet.

The Boston train leaves at 2:00 pm at 60 mph. By 3:00 pm (1 hour later), it has traveled 60 miles.

Todo #1: Estimate the distance between Boston and New York.
Todo #2: Calculate the position of the Boston train when the New York train starts.
Todo #3: Calculate the remaining distance between the two trains at 3:00 pm.
Todo #4: Calculate the relative speed of the two trains.
Todo #5: Calculate the time it takes for the trains to close the remaining distance.
Todo #6: Determine the time the trains meet.

...

Time = Distance / Speed = 155 miles / 140 mph ≈ 1.107 hours. Converting 0.107 hours to minutes: 0.107 * 60 ≈ 6.4 
minutes. So, it takes about 1 hour and 6 minutes.

Todo #1: Estimate the distance between Boston and New York.
Todo #2: Calculate the position of the Boston train when the New York train starts.
Todo #3: Calculate the remaining distance between the two trains at 3:00 pm.
Todo #4: Calculate the relative speed of the two trains.
Todo #5: Calculate the time it takes for the trains to close the remaining distance.
Todo #6: Determine the time the trains meet.

The trains start closing the gap at 3:00 pm. Adding 1 hour and 6 minutes results in a meeting time of 4:06 pm.

Todo #1: Estimate the distance between Boston and New York.
Todo #2: Calculate the position of the Boston train when the New York train starts.
Todo #3: Calculate the remaining distance between the two trains at 3:00 pm.
Todo #4: Calculate the relative speed of the two trains.
Todo #5: Calculate the time it takes for the trains to close the remaining distance.
Todo #6: Determine the time the trains meet.

The trains will meet at approximately 4:06 pm. 

This is based on the following calculation:
*   Distance Estimate: 215 miles.
*   Initial Progress: By 3:00 pm, the Boston train has traveled 60 miles (60 mph * 1 hour).
*   Remaining Distance: 215 - 60 = 155 miles.
*   Closing Speed: 60 mph + 80 mph = 140 mph.
*   Time to Meet: 155 miles / 140 mph ≈ 1.107 hours (approx. 1 hour and 6 minutes).
*   Meeting Time: 3:00 pm + 1 hour 6 minutes = 4:06 pm.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>